# Predicción de Churn en Telecomunicaciones
## Metodología CRISP-DM aplicada al dataset IBM Telco Customer Churn (Extended)

---
**Objetivo de negocio:** Identificar clientes con alta probabilidad de cancelar su suscripción antes de que ocurra el abandono, para activar campañas de retención personalizadas y reducir la tasa de churn en al menos 15%.

**Dataset:** IBM Telco Customer Churn Extended · 7,043 clientes · 33 columnas reales

**Estructura del notebook:**
1. Comprensión del negocio
2. Comprensión de los datos
3. Preparación de datos y Feature Engineering (énfasis principal)
4. Modelado
5. Evaluación
6. Conclusiones y despliegue


---
## Fase 1 — Comprensión del Negocio

### Contexto
Una empresa de telecomunicaciones ficticia enfrenta una tasa de churn del ~26%. Retener un cliente existente cuesta entre 5 y 7 veces menos que adquirir uno nuevo. El equipo de retención actúa de forma reactiva, sin anticiparse a la salida del cliente.

### Objetivos estratégicos
| Objetivo | Métrica | Meta |
|---|---|---|
| Reducir churn | Tasa de abandono mensual | ↓ 15% vs año anterior |
| Identificar churners | Recall clase positiva | ≥ 75% |
| Rentabilidad | ROI campañas retención | ≥ 3:1 |
| Calidad del modelo | ROC-AUC en test | ≥ 0.85 |

### Definición de éxito del modelo
- **Métrica principal:** ROC-AUC (maximiza discriminación general)
- **Métrica operativa:** Recall ≥ 0.75 en clase Churn=1 (preferimos falsos positivos sobre falsos negativos)
- **Umbral de decisión:** ajustable según costo de campaña de retención

### Ventajas del dataset
El dataset Extended aporta 12 columnas de alto valor estratégico:
- **`Churn Score`** — score propietario de IBM (0-100) que indica propensión al churn
- **`CLTV`** — Customer Lifetime Value calculado
- **`Churn Reason`** — razón declarada del abandono (solo para churners)
- **Geolocalización** — Latitude, Longitude, City, Zip Code



In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================
# Ejecutar si no están instaladas
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap plotly


In [2]:
# ============================================================
# IMPORTACIONES GENERALES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

print('Librerías cargadas correctamente')


Librerías cargadas correctamente


---
## Fase 2 — Comprensión de los Datos


In [8]:
# ============================================================
# 2.1 CARGA DE DATOS — IBM Telco Extended (archivo real)
# ============================================================
# Asegúrate de que el archivo esté en la misma carpeta que este notebook
df_raw = pd.read_excel('Telco_customer_churn.xlsx')

print(f'Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print()
print('Columnas disponibles:')
for i, col in enumerate(df_raw.columns):
    print(f'  {i:02d}. {col} — {df_raw[col].dtype}')

df = df_raw.copy()

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [ ]:
# ============================================================
# 2.2 ESTRUCTURA Y TIPOS DE DATOS
# ============================================================
# NOTA: El Extended usa nombres con espacios
# Ej: 'Monthly Charges' en lugar de 'MonthlyCharges'

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

print(f'Numéricas ({len(num_cols)}): {num_cols}')
print()
print(f'Categóricas ({len(cat_cols)}): {cat_cols}')
print()
df.head(3)

In [ ]:
# ============================================================
# 2.3 ANÁLISIS DE VALORES NULOS
# ============================================================
# Total Charges viene como object
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
null_df = pd.DataFrame({'Nulos': nulls, 'Porcentaje %': nulls_pct})
null_df = null_df[null_df['Nulos'] > 0].sort_values('Nulos', ascending=False)

print('=== VALORES NULOS ===')
print(null_df)
print()
print('Nota: Churn Reason tiene ~5,174 nulos porque solo aplica a clientes que SÍ hicieron churn.')
print('Esos nulos son correctos y se tratarán en feature engineering.')

In [ ]:
# ============================================================
# 2.4 DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================================
# Usamos 'Churn Value' (ya es 0/1) o 'Churn Label' (Yes/No)
churn_counts = df['Churn Label'].value_counts()
churn_pct    = df['Churn Label'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#5DCAA5', '#E8593C'], width=0.5)
axes[0].set_title('Distribución de Churn (conteo)', fontsize=13)
axes[0].set_ylabel('Número de clientes')
for i, (v, p) in enumerate(zip(churn_counts.values, churn_pct.values)):
    axes[0].text(i, v + 50, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=11)

axes[1].pie(churn_counts.values, labels=churn_counts.index,
            colors=['#5DCAA5', '#E8593C'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporción Churn vs No Churn', fontsize=13)

plt.suptitle('Desbalance de clases — tasa de churn ~26%', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 2.5 ANÁLISIS — Churn Score y CLTV
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn Score por clase
for label, color in [('No', '#5DCAA5'), ('Yes', '#E8593C')]:
    subset = df[df['Churn Label'] == label]['Churn Score']
    axes[0].hist(subset, bins=30, alpha=0.65, color=color,
                 label=f'Churn={label}', density=True)
axes[0].set_title('Distribución de Churn Score por clase', fontsize=12)
axes[0].set_xlabel('Churn Score (0-100)')
axes[0].legend()

# CLTV por clase
for label, color in [('No', '#5DCAA5'), ('Yes', '#E8593C')]:
    subset = df[df['Churn Label'] == label]['CLTV']
    axes[1].hist(subset, bins=30, alpha=0.65, color=color,
                 label=f'Churn={label}', density=True)
axes[1].set_title('Distribución de CLTV por clase', fontsize=12)
axes[1].set_xlabel('Customer Lifetime Value ($)')
axes[1].legend()

plt.suptitle('Variables exclusivas del Extended — Churn Score y CLTV', fontsize=13)
plt.tight_layout()
plt.show()

print('Churn Score promedio por clase:')
print(df.groupby('Churn Label')['Churn Score'].mean().round(1))
print()
print('CLTV promedio por clase:')
print(df.groupby('Churn Label')['CLTV'].mean().round(0))


In [ ]:
# ============================================================
# 2.5.2 ANÁLISIS EXCLUSIVO DEL EXTENDED — Churn Reason
# ============================================================
# Solo aplica a clientes que hicieron churn (el resto tiene NaN)
churn_reasons = df[df['Churn Label'] == 'Yes']['Churn Reason'].value_counts()

plt.figure(figsize=(12, 6))
bars = plt.barh(churn_reasons.index, churn_reasons.values,
                color='#E8593C', alpha=0.8, height=0.7)
plt.xlabel('Número de clientes')
plt.title('Razones de abandono declaradas (Churn Reason)', fontsize=13)
for bar, val in zip(bars, churn_reasons.values):
    plt.text(val + 1, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Insight: Las principales razones son competencia y actitud del personal de soporte.')
print('Esto orienta las estrategias de retención hacia precio y calidad de atención.')


In [ ]:
# ============================================================
# 2.5.3 EDA — VARIABLES CATEGÓRICAS CLAVE
# ============================================================
cat_features_eda = ['Contract', 'Payment Method', 'Internet Service']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, cat_features_eda):
    churn_rate = df.groupby(col)['Churn Value'].mean() * 100
    churn_rate = churn_rate.sort_values(ascending=False)
    
    bars = ax.barh(churn_rate.index, churn_rate.values,
                   color='#E8593C', alpha=0.75)
    ax.set_xlabel('Tasa de churn (%)')
    ax.set_title(f'Churn % por {col}', fontsize=12)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    for bar, val in zip(bars, churn_rate.values):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

plt.suptitle('Tasa de churn por variable categórica', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2.6 EDA — VARIABLES NUMÉRICAS
# ============================================================
num_features_eda = ['Tenure Months', 'Monthly Charges', 'Total Charges',
                    'Churn Score', 'CLTV']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, col in zip(axes, num_features_eda):
    for label, color in [('No', '#5DCAA5'), ('Yes', '#E8593C')]:
        subset = df[df['Churn Label'] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f'Churn={label}', density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Distribución de variables numéricas por Churn', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2.7 MATRIZ DE CORRELACIÓN
# ============================================================
num_for_corr = ['Tenure Months', 'Monthly Charges', 'Total Charges',
                'Churn Score', 'CLTV', 'Churn Value']

corr_df = df[num_for_corr].copy()
corr_df['Total Charges'] = pd.to_numeric(corr_df['Total Charges'], errors='coerce')

corr_matrix = corr_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(9, 6))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, annot_kws={'size': 10})
plt.title('Matriz de correlación — variables numéricas', fontsize=13)
plt.tight_layout()
plt.show()

print('=== Correlación con Churn Value ===')
print(corr_matrix['Churn Value'].drop('Churn Value').sort_values(key=abs, ascending=False))


---
## Fase 3 — Preparación de Datos y Feature Engineering

Esta es la fase más extensa. 
División:
1. Limpieza base y renombrado
2. Variable objetivo
3. Feature engineering de comportamiento temporal
4. Feature engineering de adopción de servicios
5. Feature engineering de score de riesgo compuesto
6. Feature engineering de interacciones
7. Feature engineering desde Churn Reason (exclusivo Extended)
8. Encoding de categóricas
9. Tratamiento del desbalance
10. Split y escalado


In [ ]:
# ============================================================
# 3.1 LIMPIEZA BASE
# ============================================================
df_clean = df.copy()

# Convertir Total Charges a numérico
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce')
df_clean['Total Charges'] = df_clean['Total Charges'].fillna(0)

# Eliminar columnas no predictivas o redundantes
# - CustomerID: identificador
# - Count: siempre vale 1
# - Country, State: sin variabilidad (todos California/USA)
# - Lat Long: ya tenemos Latitude y Longitude por separado
# - Churn Label: misma info que Churn Value (nuestro target)
# - Churn Reason: se usará para crear features, luego se elimina
cols_drop = ['CustomerID', 'Count', 'Country', 'State', 'Lat Long', 'Churn Label']
df_clean.drop(columns=cols_drop, inplace=True)

print(f'Shape después de limpieza: {df_clean.shape}')
print(f'Nulos restantes (sin Churn Reason): {df_clean.drop(columns="Churn Reason").isnull().sum().sum()}')



In [ ]:
# ============================================================
# 3.2 VARIABLE OBJETIVO
# ============================================================
# 'Churn Value' ya viene como 0/1 en el Extended — no requiere mapeo
print('Distribución Churn Value:')
print(df_clean['Churn Value'].value_counts())
print(f'Tasa de churn: {df_clean["Churn Value"].mean()*100:.1f}%')


In [ ]:
# ============================================================
# 3.3 FEATURE ENGINEERING — COMPORTAMIENTO TEMPORAL
# ============================================================

# 1. Segmento de antigüedad
df_clean['tenure_cohort'] = pd.cut(
    df_clean['Tenure Months'],
    bins=[0, 12, 24, 48, 72],
    labels=[0, 1, 2, 3],
    include_lowest=True
).astype(int)

# 2. Gasto mensual promedio real
df_clean['avg_monthly_spend'] = np.where(
    df_clean['Tenure Months'] > 0,
    df_clean['Total Charges'] / df_clean['Tenure Months'],
    df_clean['Monthly Charges']
).round(2)

# 3. Drift entre cargo actual y promedio histórico
df_clean['charge_drift'] = (df_clean['Monthly Charges'] - df_clean['avg_monthly_spend']).round(2)

# 4. Meses hasta fin de contrato estimado
contract_duration_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df_clean['contract_duration_months'] = df_clean['Contract'].map(contract_duration_map)
df_clean['months_to_contract_end'] = np.maximum(
    df_clean['contract_duration_months'] - (df_clean['Tenure Months'] % df_clean['contract_duration_months']),
    0
)

# 5. Cliente nuevo (primeros 6 meses)
df_clean['is_new_customer'] = (df_clean['Tenure Months'] <= 6).astype(int)

# 6. Cuadrado de tenure (efecto no lineal de fidelidad)
df_clean['tenure_squared'] = df_clean['Tenure Months'] ** 2

# 7. Log de Total Charges (reduce asimetría)
df_clean['log_total_charges'] = np.log1p(df_clean['Total Charges']).round(4)

print('Features temporales creadas')


In [ ]:
# ============================================================
# 3.4 FEATURE ENGINEERING — ADOPCIÓN DE SERVICIOS
# ============================================================
yes_no_map = {'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0}

# NOTA: en el Extended los nombres tienen espacios
service_cols = [
    'Online Security', 'Online Backup', 'Device Protection',
    'Tech Support', 'Streaming TV', 'Streaming Movies', 'Multiple Lines'
]

for col in service_cols:
    df_clean[col + '_bin'] = df_clean[col].map(yes_no_map).fillna(0).astype(int)

# Número total de servicios adicionales
df_clean['num_additional_services'] = df_clean[[c + '_bin' for c in service_cols]].sum(axis=1)

# Ratio de adopción (0-1)
df_clean['service_adoption_rate'] = (df_clean['num_additional_services'] / 7).round(3)

# Tiene servicios de seguridad
df_clean['has_security_services'] = (
    (df_clean['Online Security_bin'] == 1) |
    (df_clean['Online Backup_bin'] == 1) |
    (df_clean['Device Protection_bin'] == 1)
).astype(int)

# Solo streaming (riesgo de irse a Netflix/otros)
df_clean['streaming_only'] = (
    (df_clean['Streaming TV_bin'] == 1) |
    (df_clean['Streaming Movies_bin'] == 1)
).astype(int)

print(' Features de servicios creadas')


In [ ]:
# ============================================================
# 3.5 FEATURE ENGINEERING — SCORE DE RIESGO COMPUESTO
# ============================================================
# DIFERENCIA vs versión anterior: ahora usamos Churn Score real de IBM
# en lugar del SatisfactionScore simulado. Mucho más confiable.

def minmax_norm(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

risk_high_churn_score    = minmax_norm(df_clean['Churn Score'])            # ya es señal directa
risk_high_monthly_charge = minmax_norm(df_clean['Monthly Charges'])
risk_short_tenure        = minmax_norm(1 / (df_clean['Tenure Months'] + 1))
risk_month_to_month      = (df_clean['Contract'] == 'Month-to-month').astype(float)
risk_low_services        = minmax_norm(1 - df_clean['service_adoption_rate'])
risk_new_customer        = df_clean['is_new_customer'].astype(float)

df_clean['churn_risk_score'] = (
    0.30 * risk_high_churn_score    +  # Churn Score IBM es el predictor más fuerte
    0.20 * risk_month_to_month      +
    0.20 * risk_short_tenure        +
    0.15 * risk_high_monthly_charge +
    0.10 * risk_low_services        +
    0.05 * risk_new_customer
).round(4)

df_clean['risk_segment'] = pd.cut(
    df_clean['churn_risk_score'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Bajo', 'Medio', 'Alto'],
    include_lowest=True
)

# Validar score vs churn real
seg_churn = df_clean.groupby('risk_segment', observed=True)['Churn Value'].mean() * 100

plt.figure(figsize=(8, 4))
bars = plt.bar(seg_churn.index, seg_churn.values,
               color=['#5DCAA5', '#EF9F27', '#E8593C'], width=0.5)
plt.title('Tasa de churn real por segmento de riesgo compuesto', fontsize=12)
plt.ylabel('Tasa de churn (%)')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, seg_churn.values):
    plt.text(bar.get_x() + bar.get_width()/2, v + 0.5,
             f'{v:.1f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

print('churn_risk_score creado usando Churn Score real de IBM')



In [ ]:
# ============================================================
# 3.6 FEATURE ENGINEERING — INTERACCIONES Y RATIOS
# ============================================================

# 1. Gasto por servicio adicional (valor percibido)
df_clean['spend_per_service'] = np.where(
    df_clean['num_additional_services'] > 0,
    df_clean['Monthly Charges'] / (df_clean['num_additional_services'] + 1),
    df_clean['Monthly Charges']
).round(2)

# 2. CLTV por mes de antigüedad
df_clean['cltv_per_month'] = np.where(
    df_clean['Tenure Months'] > 0,
    df_clean['CLTV'] / df_clean['Tenure Months'],
    df_clean['CLTV']
).round(2)

# 3. Senior Citizen en el Extended es 'Yes'/'No' (texto), no 0/1
df_clean['Senior Citizen_bin'] = (df_clean['Senior Citizen'] == 'Yes').astype(int)

# 4. Senior + contrato mensual (grupo vulnerable)
df_clean['senior_month_to_month'] = (
    (df_clean['Senior Citizen_bin'] == 1) &
    (df_clean['Contract'] == 'Month-to-month')
).astype(int)

# 5. Sin soporte técnico + cargo alto
df_clean['no_support_high_charge'] = (
    (df_clean['Tech Support_bin'] == 0) &
    (df_clean['Monthly Charges'] > df_clean['Monthly Charges'].median())
).astype(int)

# 6. Churn Score × Monthly Charges (riesgo financiero ponderado)
df_clean['risk_charge_product'] = (
    minmax_norm(df_clean['Churn Score']) * minmax_norm(df_clean['Monthly Charges'])
).round(4)

# 7. CLTV alto + churn score alto = cliente valioso en riesgo (prioridad máxima)
df_clean['high_value_at_risk'] = (
    (df_clean['CLTV'] > df_clean['CLTV'].quantile(0.75)) &
    (df_clean['Churn Score'] > df_clean['Churn Score'].quantile(0.75))
).astype(int)

print('Features de interacción creadas')
print(f'  Clientes de alto valor en riesgo: {df_clean["high_value_at_risk"].sum():,}')

In [ ]:
# ============================================================
# 3.7 FEATURE ENGINEERING — CHURN REASON
# ============================================================
# Churn Reason es NaN para clientes que NO hicieron churn.
# No podemos usarla directamente como predictor (data leakage).
# Estrategia: crear categorías de razón que reflejen el TIPO de riesgo.

# Mapear razones a categorías de riesgo
def categorize_churn_reason(reason):
    if pd.isna(reason):
        return 'no_churn'
    reason = reason.lower()
    if 'competitor' in reason:
        return 'competitor'
    elif 'price' in reason or 'charges' in reason or 'extra' in reason:
        return 'price'
    elif 'attitude' in reason or 'service' in reason or 'support' in reason:
        return 'service_quality'
    elif 'moved' in reason or 'deceased' in reason:
        return 'external'
    elif 'product' in reason or 'dissatisfaction' in reason:
        return 'product'
    else:
        return 'other'

df_clean['churn_reason_category'] = df_clean['Churn Reason'].apply(categorize_churn_reason)

print('Distribución de categorías de razón de churn:')
print(df_clean['churn_reason_category'].value_counts())
print()
print('NOTA: Esta columna se usa SOLO para análisis descriptivo,')
print('no como feature del modelo (sería data leakage).')

# Visualización
reason_churn = df_clean[df_clean['Churn Value'] == 1]['churn_reason_category'].value_counts()
plt.figure(figsize=(10, 4))
plt.bar(reason_churn.index, reason_churn.values,
        color=['#E8593C', '#EF9F27', '#7F77DD', '#5DCAA5', '#1D9E75', '#999'])
plt.title('Categorías de razón de churn', fontsize=12)
plt.ylabel('Número de clientes')
for i, v in enumerate(reason_churn.values):
    plt.text(i, v + 2, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

# Eliminar columna original (ya extrajimos lo útil)
df_clean.drop(columns=['Churn Reason'], inplace=True)


In [ ]:
# ============================================================
# 3.8 RESUMEN DE FEATURES CREADAS
# ============================================================
engineered_features = [
    ('tenure_cohort',           'Temporal',          'Segmento de antigüedad del cliente (0=nuevo, 3=leal)'),
    ('avg_monthly_spend',       'Temporal',          'Gasto mensual promedio real (Total/Tenure)'),
    ('charge_drift',            'Temporal',          'Diferencia cargo actual vs histórico'),
    ('months_to_contract_end',  'Temporal',          'Meses hasta vencimiento de contrato'),
    ('is_new_customer',         'Temporal',          'Cliente con menos de 6 meses'),
    ('tenure_squared',          'Transformación',    'Efecto no lineal de la fidelidad'),
    ('log_total_charges',       'Transformación',    'Reduce asimetría en cargos totales'),
    ('num_additional_services', 'Servicios',         'Total de servicios adicionales contratados'),
    ('service_adoption_rate',   'Servicios',         'Ratio de adopción (0-1)'),
    ('has_security_services',   'Servicios',         'Tiene al menos un servicio de seguridad'),
    ('streaming_only',          'Servicios',         'Solo usa servicios de streaming'),
    ('churn_risk_score',        'Riesgo compuesto',  'Score ponderado con Churn Score IBM real'),
    ('spend_per_service',       'Interacción',       'Costo por servicio adicional'),
    ('cltv_per_month',          'Interacción',       'CLTV dividido entre antigüedad'),
    ('senior_month_to_month',   'Interacción',       'Senior Citizen con contrato mensual'),
    ('no_support_high_charge',  'Interacción',       'Sin soporte técnico y cargo alto'),
    ('risk_charge_product',     'Interacción',       'Churn Score × Monthly Charges normalizado'),
    ('high_value_at_risk',      'Interacción',       'CLTV alto + Churn Score alto (prioridad máxima)'),
]

summary_df = pd.DataFrame(engineered_features,
                           columns=['Feature', 'Categoría', 'Descripción / Hipótesis'])
print(f'=== RESUMEN FEATURE ENGINEERING — {len(engineered_features)} features creadas ===')
print(summary_df.to_string(index=False))


In [ ]:
# ============================================================
# 3.9 ENCODING DE VARIABLES CATEGÓRICAS
# ============================================================
df_model = df_clean.copy()

# Binarias directas
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
binary_cols = ['Gender', 'Partner', 'Dependents', 'Phone Service', 'Paperless Billing']
for col in binary_cols:
    df_model[col] = df_model[col].map(binary_map)

# One-Hot Encoding para categóricas con múltiples valores
ohe_cols = ['Contract', 'Payment Method', 'Internet Service']
df_model = pd.get_dummies(df_model, columns=ohe_cols, drop_first=False, dtype=int)

# Eliminar columnas ya procesadas o que no van al modelo
cols_to_drop = (
    service_cols +                        # reemplazadas por _bin
    ['Senior Citizen',                    # reemplazada por Senior Citizen_bin
     'risk_segment',                      # categórica derivada de churn_risk_score
     'churn_reason_category',             # no es predictor (data leakage)
     'contract_duration_months',          # auxiliar del cálculo
     'City', 'Zip Code',                  # demasiada cardinalidad
     'Latitude', 'Longitude']             # coordenadas sin procesar
)
df_model.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f'Shape tras encoding: {df_model.shape}')
print(f'Total features para el modelo: {df_model.shape[1] - 1}')
df_model.head(2)


In [ ]:
# ============================================================
# 3.10 SPLIT TRAIN / TEST
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_model.drop(columns=['Churn Value'])
y = df_model['Churn Value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas')
print(f'Churn en train: {y_train.mean()*100:.1f}% | Churn en test: {y_test.mean()*100:.1f}%')

In [ ]:
# ============================================================
# 3.11 TRATAMIENTO DEL DESBALANCE — SMOTE
# ============================================================
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Antes  — Churn: {y_train.sum():,} | No Churn: {(y_train==0).sum():,}')
print(f'Después — Churn: {y_train_res.sum():,} | No Churn: {(y_train_res==0).sum():,}')



In [ ]:
# ============================================================
# 3.12 ESCALADO
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print('Escalado completado')



---
## Fase 4 — Modelado

Comparamos 3 modelos:
1. **Regresión Logística** — línea base interpretable
2. **Random Forest** — ensemble robusto
3. **XGBoost** — gradient boosting de alto rendimiento


In [ ]:
# ============================================================
# 4.1 ENTRENAMIENTO DE LOS 3 MODELOS
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                              precision_score, accuracy_score,
                              classification_report, confusion_matrix,
                              roc_curve, precision_recall_curve)

# Modelo 1: Regresión Logística (requiere datos escalados)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_res)

# Modelo 2: Random Forest (no requiere escalado)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf.fit(X_train_res, y_train_res)

# Modelo 3: XGBoost
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train_res, y_train_res,
        eval_set=[(X_test, y_test)], verbose=False)

print('✅ Modelos entrenados')


In [ ]:
# ============================================================
# 4.2 EVALUACIÓN COMPARATIVA
# ============================================================
def evaluate_model(model, X_test, y_test, model_name, scaled=False):
    X = X_test_scaled if scaled else X_test
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    
    return {
        'Modelo': model_name,
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
        'F1':      round(f1_score(y_test, y_pred), 4),
        'Recall':  round(recall_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        '_proba': y_proba,
        '_pred':  y_pred
    }

results = [
    evaluate_model(lr,  X_test, y_test, 'Regresión Logística', scaled=True),
    evaluate_model(rf,  X_test, y_test, 'Random Forest'),
    evaluate_model(xgb, X_test, y_test, 'XGBoost'),
]

results_df = pd.DataFrame(results).drop(columns=['_proba', '_pred'])
print('=== COMPARACIÓN DE MODELOS ===')
print(results_df.to_string(index=False))
print()

# Identificar mejor modelo por ROC-AUC
best_idx = results_df['ROC-AUC'].idxmax()
print(f'🏆 Mejor modelo: {results_df.loc[best_idx, "Modelo"]} '
      f'(ROC-AUC = {results_df.loc[best_idx, "ROC-AUC"]})')


In [ ]:
# ============================================================
# 4.3 CURVAS ROC
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = ['#7F77DD', '#1D9E75', '#E8593C']
models_info = [
    (lr,  'Regresión Logística', True),
    (rf,  'Random Forest', False),
    (xgb, 'XGBoost', False)
]

# Curva ROC
for (model, name, scaled), color in zip(models_info, colors):
    X = X_test_scaled if scaled else X_test
    y_proba = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Aleatorio')
axes[0].fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (Recall)')
axes[0].set_title('Curva ROC — Comparación de modelos', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Curva Precision-Recall (más informativa con desbalance)
for (model, name, scaled), color in zip(models_info, colors):
    X = X_test_scaled if scaled else X_test
    y_proba = model.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(rec, prec, color=color, lw=2, label=name)

baseline = y_test.mean()
axes[1].axhline(baseline, color='gray', linestyle='--', alpha=0.6,
                label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4.4 MATRICES DE CONFUSIÓN
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (model, name, scaled) in zip(axes, models_info):
    X = X_test_scaled if scaled else X_test
    y_pred = model.predict(X)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='Blues', linewidths=0.5,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de Confusión en datos de prueba', fontsize=13)
plt.tight_layout()
plt.show()

# Reporte detallado del mejor modelo (XGBoost)
print('=== REPORTE DETALLADO — XGBoost ===')
print(classification_report(y_test, xgb.predict(X_test),
                             target_names=['No Churn', 'Churn']))


In [ ]:
# ============================================================
# 4.5 IMPORTANCIA DE FEATURES — Random Forest y XGBoost
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top_n = 20

for ax, model, name in [
    (axes[0], rf,  'Random Forest'),
    (axes[1], xgb, 'XGBoost')
]:
    importances = pd.Series(
        model.feature_importances_, index=X_train_res.columns
    ).nlargest(top_n).sort_values()
    
    colors_imp = ['#E8593C' if 'churn_risk' in i or 'satisfaction' in i.lower()
                  else '#7F77DD' if any(k in i for k in ['tenure', 'charge', 'contract'])
                  else '#1D9E75'
                  for i in importances.index]
    
    ax.barh(importances.index, importances.values, color=colors_imp, height=0.65)
    ax.set_title(f'Top {top_n} features — {name}', fontsize=12)
    ax.set_xlabel('Importancia')
    
    # Leyenda de colores
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(color='#E8593C', label='Satisfacción / Riesgo'),
        Patch(color='#7F77DD', label='Temporales / Contrato'),
        Patch(color='#1D9E75', label='Servicios / Otros')
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.suptitle('Importancia de Features por Modelo', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4.6 ANÁLISIS SHAP — INTERPRETABILIDAD (XGBoost)
# ============================================================
try:
    import shap
    
    explainer   = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values, X_test,
        plot_type='bar',
        max_display=20,
        show=False
    )
    plt.title('SHAP — Importancia global de features (XGBoost)', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values, X_test,
        max_display=15,
        show=False
    )
    plt.title('SHAP — Dirección e impacto por feature', fontsize=13)
    plt.tight_layout()
    plt.show()

except ImportError:
    print('SHAP no disponible. Instalar con: pip install shap')


In [ ]:
# ============================================================
# 4.7 OPTIMIZACIÓN DE UMBRAL DE DECISIÓN
# ============================================================
# En problemas de churn, preferimos mayor recall (no perder churners)
# aunque baje la precisión. El umbral óptimo depende del costo de intervención.

y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.01)
recalls, precisions, f1s = [], [], []

for t in thresholds:
    y_pred_t = (y_proba_xgb >= t).astype(int)
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

optimal_t = thresholds[np.argmax(f1s)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, recalls,    color='#E8593C', lw=2, label='Recall')
ax.plot(thresholds, precisions, color='#7F77DD', lw=2, label='Precision')
ax.plot(thresholds, f1s,        color='#1D9E75', lw=2, label='F1-Score')
ax.axvline(optimal_t, color='gray', linestyle='--', alpha=0.8,
           label=f'Umbral óptimo F1: {optimal_t:.2f}')
ax.axvline(0.5, color='black', linestyle=':', alpha=0.4, label='Default (0.5)')
ax.set_xlabel('Umbral de decisión')
ax.set_ylabel('Métrica')
ax.set_title('Recall, Precision y F1 según umbral — XGBoost', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Métricas con umbral óptimo
y_pred_optimal = (y_proba_xgb >= optimal_t).astype(int)
print(f'=== Métricas con umbral = {optimal_t:.2f} ===')
print(f'Recall:    {recall_score(y_test, y_pred_optimal):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_optimal):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_optimal):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.4f}')


---
## Fase 5 — Evaluación desde la perspectiva de negocio


In [ ]:
# ============================================================
# 5.1 ANÁLISIS COSTO-BENEFICIO
# ============================================================
# Supuestos de negocio (ajustar según empresa real)
MONTHLY_REVENUE_PER_CUSTOMER = 65    # USD promedio
AVG_TENURE_MONTHS             = 24   # meses promedio de vida útil
CUSTOMER_LTV                  = MONTHLY_REVENUE_PER_CUSTOMER * AVG_TENURE_MONTHS
RETENTION_CAMPAIGN_COST       = 50   # USD por cliente contactado
RETENTION_SUCCESS_RATE        = 0.30  # 30% de clientes contactados retienen

cm = confusion_matrix(y_test, y_pred_optimal)
TN, FP, FN, TP = cm.ravel()

# Beneficio: clientes retenidos correctamente identificados
benefit = TP * RETENTION_SUCCESS_RATE * CUSTOMER_LTV

# Costo: campañas enviadas (TP + FP)
cost    = (TP + FP) * RETENTION_CAMPAIGN_COST

# Pérdida por no detectar churners reales (FN)
missed_revenue = FN * MONTHLY_REVENUE_PER_CUSTOMER * 6  # 6 meses de ingreso perdido

roi = (benefit - cost) / cost if cost > 0 else 0

print('=== ANÁLISIS COSTO-BENEFICIO (datos de test escalados a 12 meses) ===')
print(f'Clientes en riesgo detectados (TP):   {TP:,}')
print(f'Falsas alarmas (FP):                  {FP:,}')
print(f'Churners no detectados (FN):          {FN:,}')
print()
print(f'Beneficio estimado (retención):       ${benefit:,.0f}')
print(f'Costo de campañas:                    ${cost:,.0f}')
print(f'Ingreso perdido por FN:               ${missed_revenue:,.0f}')
print(f'Beneficio neto:                       ${benefit - cost:,.0f}')
print(f'ROI campañas:                         {roi:.1f}x')
print()
print(f'Meta ROI ≥ 3:1 → {"✅ ALCANZADA" if roi >= 3 else "❌ NO alcanzada"}')


In [ ]:
# ============================================================
# 5.2 CURVA DE GANANCIA (LIFT CURVE)
# ============================================================
# ¿Cuánto mejor que el azar es nuestro modelo?

# Ordenar por probabilidad descendente
gain_df = pd.DataFrame({
    'y_real':  y_test.values,
    'y_proba': y_proba_xgb
}).sort_values('y_proba', ascending=False).reset_index(drop=True)

gain_df['cumulative_churn']  = gain_df['y_real'].cumsum()
gain_df['pct_population']    = (gain_df.index + 1) / len(gain_df) * 100
gain_df['pct_churn_captured'] = gain_df['cumulative_churn'] / gain_df['y_real'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Curva de Ganancia
axes[0].plot(gain_df['pct_population'], gain_df['pct_churn_captured'],
             color='#7F77DD', lw=2.5, label='XGBoost')
axes[0].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Modelo aleatorio')
axes[0].fill_between(gain_df['pct_population'],
                     gain_df['pct_churn_captured'],
                     gain_df['pct_population'],
                     alpha=0.15, color='#7F77DD')

# Línea de referencia: con 30% de clientes contactados capturamos X% de churners
idx_30 = gain_df[gain_df['pct_population'] <= 30].index[-1]
captured_30 = gain_df.loc[idx_30, 'pct_churn_captured']
axes[0].annotate(f'Top 30% contactados\ncaptura {captured_30:.0f}% de churners',
                 xy=(30, captured_30), xytext=(45, captured_30 - 10),
                 fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))

axes[0].set_xlabel('% de clientes contactados')
axes[0].set_ylabel('% de churners capturados')
axes[0].set_title('Curva de Ganancia — XGBoost', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Lift Chart
gain_df['lift'] = gain_df['pct_churn_captured'] / gain_df['pct_population']
axes[1].plot(gain_df['pct_population'], gain_df['lift'],
             color='#E8593C', lw=2.5)
axes[1].axhline(1, color='gray', linestyle='--', alpha=0.6, label='Lift = 1 (aleatorio)')
axes[1].set_xlabel('% de clientes contactados')
axes[1].set_ylabel('Lift')
axes[1].set_title('Curva de Lift — XGBoost', fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Análisis de Ganancia y Lift', fontsize=13)
plt.tight_layout()
plt.show()

lift_30 = gain_df.loc[idx_30, 'lift']
print(f'\nContactando al 30% de mayor riesgo:')
print(f'  → Se captura el {captured_30:.0f}% de todos los churners')
print(f'  → Lift = {lift_30:.2f}x mejor que selección aleatoria')


In [ ]:
# ============================================================
# 5.3 VALIDACIÓN DE KPIs DE NEGOCIO
# ============================================================
kpis = {
    'ROC-AUC ≥ 0.85':    roc_auc_score(y_test, y_proba_xgb) >= 0.85,
    'Recall ≥ 0.75':     recall_score(y_test, y_pred_optimal) >= 0.75,
    'ROI campañas ≥ 3x': roi >= 3,
    'Lift top 30% ≥ 2x': lift_30 >= 2
}

print('=== VALIDACIÓN DE KPIs DE NEGOCIO ===')
for kpi, met in kpis.items():
    status = '✅ CUMPLIDO' if met else '❌ NO cumplido'
    print(f'  {kpi}: {status}')

print(f'\nKPIs alcanzados: {sum(kpis.values())}/{len(kpis)}')


---
## Fase 6 — Conclusiones y Despliegue


In [ ]:
# ============================================================
# 6.1 PREDICCIÓN SOBRE NUEVOS CLIENTES (simulación de despliegue)
# ============================================================
import warnings
warnings.filterwarnings('ignore')

def predict_churn_risk(new_customers_df):
    """
    Función de inferencia lista para integrar en una API.
    Recibe un DataFrame con las mismas columnas que X_train.
    Retorna probabilidad de churn y segmento de riesgo.
    """
    proba = xgb.predict_proba(new_customers_df)[:, 1]
    risk  = pd.cut(proba,
                   bins=[0, 0.33, 0.66, 1.0],
                   labels=['Bajo', 'Medio', 'Alto'],
                   include_lowest=True)
    return pd.DataFrame({
        'churn_probability': proba.round(4),
        'risk_segment':      risk,
        'recommend_action':  pd.cut(proba,
                                    bins=[0, 0.33, 0.66, 1.0],
                                    labels=['Monitoreo', 'Oferta proactiva', 'Intervención urgente'],
                                    include_lowest=True)
    })

# Predicción sobre 10 clientes del set de prueba
sample = X_test.head(10)
predictions = predict_churn_risk(sample)
predictions['churn_real'] = y_test.head(10).values

print('=== PREDICCIONES SOBRE MUESTRA DE CLIENTES ===')
print(predictions.to_string(index=True))


In [ ]:
# ============================================================
# 6.2 GUARDAR MODELO FINAL
# ============================================================
import pickle

# Guardar modelo XGBoost y scaler
with open('xgb_churn_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

with open('scaler_churn.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Guardar lista de features para validación en producción
feature_names = X_train_res.columns.tolist()
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print('✅ Modelo guardado: xgb_churn_model.pkl')
print('✅ Scaler guardado: scaler_churn.pkl')
print(f'✅ Features: {len(feature_names)} variables')


In [ ]:
# ============================================================
# 6.3 RESUMEN EJECUTIVO FINAL
# ============================================================
print('=' * 65)
print('          RESUMEN EJECUTIVO — PROYECTO CHURN CRISP-DM')
print('=' * 65)

print()
print('📊 DATOS')
print(f'   Dataset:   IBM Telco Customer Churn Extended')
print(f'   Registros: {len(df_model):,} clientes')
print(f'   Features originales: 21 → Features finales: {len(feature_names)}')
print(f'   Tasa de churn base: ~26%')

print()
print('🔧 FEATURE ENGINEERING')
print('   18 features nuevas creadas en 4 categorías:')
print('   • 5 temporales (tenure_cohort, charge_drift, ...)')
print('   • 4 de servicios (num_additional_services, ...)')
print('   • 1 score compuesto (churn_risk_score)')
print('   • 8 interacciones y transformaciones')

print()
print('🤖 MEJOR MODELO: XGBoost')
print(f'   ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.4f}')
print(f'   Recall:    {recall_score(y_test, y_pred_optimal):.4f}')
print(f'   Precision: {precision_score(y_test, y_pred_optimal):.4f}')
print(f'   F1-Score:  {f1_score(y_test, y_pred_optimal):.4f}')
print(f'   Umbral óptimo: {optimal_t:.2f}')

print()
print('💼 IMPACTO DE NEGOCIO')
print(f'   ROI campañas: {roi:.1f}x')
print(f'   Beneficio neto estimado: ${benefit - cost:,.0f}')
print(f'   Contactando top 30% se captura {captured_30:.0f}% de churners')
print(f'   Lift: {lift_30:.2f}x mejor que campaña aleatoria')

print()
print('✅ KPIs alcanzados:', sum(kpis.values()), '/', len(kpis))
print('=' * 65)


---
## Referencias

- IBM Sample Data Sets: [IBM Telco Customer Churn](https://www.ibm.com/docs/en/cognos-analytics/12.0.x?topic=samples-telco-customer-churn)
- Kaggle Extended: [ylchang/telco-customer-churn-1113](https://www.kaggle.com/datasets/ylchang/telco-customer-churn-1113)
- CRISP-DM Reference Guide: [crisp-dm.eu](https://www.crisp-dm.eu)
- SMOTE: Chawla et al. (2002). *SMOTE: Synthetic Minority Over-sampling Technique*
- XGBoost: Chen & Guestrin (2016). *XGBoost: A Scalable Tree Boosting System*
- SHAP: Lundberg & Lee (2017). *A Unified Approach to Interpreting Model Predictions*
